# CIF/mmCIF → PDB-Konverter

Kleines Colab-Hilfsnotebook für das KI-Strukturmodell-Labor.

Ziel: AF3-/AlphaFold-Server-Ergebnisse als `.cif` oder `.mmcif` hochladen und als `.pdb` herunterladen. Die Originaldateien sollten zusätzlich aufgehoben werden.


## 1. Gemmi installieren

Gemmi ist eine robuste Bibliothek für makromolekulare Strukturformate.


In [ ]:
!pip -q install gemmi
import gemmi, pathlib, zipfile, os
from google.colab import files
print('Gemmi-Version:', gemmi.__version__)


## 2. CIF/mmCIF-Dateien hochladen

Du kannst eine oder mehrere Dateien gleichzeitig hochladen.


In [ ]:
uploaded = files.upload()
input_files = [pathlib.Path(name) for name in uploaded.keys()]
print('Hochgeladen:')
for p in input_files:
    print('-', p)


## 3. Nach PDB konvertieren

Für das KI-Strukturmodell-Labor genügt normalerweise das Standard-PDB. Bei sehr großen Strukturen kann das PDB-Format Grenzen haben; für Calmodulin ist das unkritisch.


In [ ]:
output_files = []
for p in input_files:
    if p.suffix.lower() not in ['.cif', '.mmcif']:
        print('Übersprungen, keine CIF/mmCIF-Datei:', p)
        continue
    structure = gemmi.read_structure(str(p))
    structure.setup_entities()
    out = p.with_suffix('.pdb')
    structure.write_pdb(str(out))
    output_files.append(out)
    print(f'Konvertiert: {p} → {out}')

if not output_files:
    print('Keine Ausgabedateien erzeugt.')
elif len(output_files) == 1:
    files.download(str(output_files[0]))
else:
    zip_name = 'converted_pdb_files.zip'
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as z:
        for out in output_files:
            z.write(out, arcname=out.name)
    files.download(zip_name)


## 4. Zielname im Repo

Für das Calmodulin-AF3-Modell im KI-Strukturmodell-Labor wird empfohlen:

```text
structures/calmodulin/af3_ca_model.pdb
```

Bei mehreren AF3-Ergebnissen zuerst alle ansehen und dann das didaktisch passendste Modell unter diesem Namen ablegen.
